# TRUJAMAn 🧙 · 2. Test

**T**extual **R**ecognition **U**nit for **J**udeo-**A**rabic (and Hebrew) in **M**edieval **A**nthologies · *beta v0.1.0*

This notebook tests the **published model** (`trujaman_v0.1-beta.joblib`) on the 40 works held out in `1.Data/b.Training/b.2.Evaluation/`. It does not retrain anything. Like `1.Training.ipynb`, it needs the FJMS texts, which are not distributed with this repository.

1. Accuracy per work and per author
2. Precision and recall for Hebrew and Judeo-Arabic
3. Held-out accuracy: authors seen and unseen in training
4. Confusion matrix
5. Errors at language switches
6. Confidence
7. Most frequent words: mislabelled, Judeo-Arabic, Hebrew
8. Most frequent chunks: mislabelled, Judeo-Arabic, Hebrew
9. Words analysis: errors, Judeo-Arabic, Hebrew

The outputs show counts and scores only, plus single words and chunks, and short five-word examples in sections 7 and 8: never longer running text.

In [1]:
# ===========================================================================
EVAL_DIR   = "1.Data/b.Training/b.2.Evaluation"   # held-out works
TRAIN_DIR  = "1.Data/b.Training/b.1.Training"     # read only for word counts and authors
MODEL_PATH = "trujaman_v0.1-beta.joblib"
# ===========================================================================

import re, glob, os
from collections import Counter
import numpy as np, pandas as pd, joblib
from IPython.display import Markdown, display
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

pd.set_option("display.width", 140)

# Same cleaning and features as 1.Training.ipynb: change both or neither.
# 05C4 upper dot | 05BF rafe | 05F3/05F4 geresh, gershayim | ' " | punctuation
STRIP = re.compile("[\u05c4\u05bf\u05f3\u05f4\u0027\u0022,.;:()\\[\\]\\-\u2013\u2014?!\u2026]")


def normalise(s):
    return STRIP.sub("", str(s))


def load_dir(folder):
    """Read every work CSV in a folder into one frame, tagged by source file."""
    paths = sorted(p for p in glob.glob(os.path.join(folder, "*.csv"))
                   if os.path.basename(p) not in ("works.csv", "evaluation_scores.csv"))
    if not paths:
        raise FileNotFoundError(
            f"No texts found in {folder}. The FJMS texts are not distributed with "
            "this repository: see 1.Training.ipynb on how to add them.")
    frames = []
    for p in paths:
        d = pd.read_csv(p)
        d = d[d.lang.isin(["JA", "HE"])].copy()
        d["work"] = os.path.splitext(os.path.basename(p))[0]
        d["doc"] = d.work + ":" + d.wID.astype(str).str.split(".").str[0]
        d["w"] = d.word.map(normalise)
        frames.append(d[d.w.str.len() > 0])
    return pd.concat(frames, ignore_index=True)


def features(d):
    """Target word marked with ~, flanked by its neighbours on the same page."""
    prev = d.w.groupby(d.doc).shift(1).fillna("")
    nxt = d.w.groupby(d.doc).shift(-1).fillna("")
    return (prev + " ~" + d.w + "~ " + nxt).values


def scores(x):
    """Accuracy, precision, recall and F1 for one group of words."""
    y, p = (x.lang == "HE").astype(int), (x.pred == "HE").astype(int)
    return pd.Series({
        "works": x.work.nunique() if "work" in x else 1, "words": len(x), "pct_HE": y.mean(),
        "accuracy": accuracy_score(y, p),
        "P_HE": precision_score(y, p, zero_division=0), "R_HE": recall_score(y, p, zero_division=0),
        "F1_HE": f1_score(y, p, zero_division=0),
        "P_JA": precision_score(y, p, pos_label=0, zero_division=0),
        "R_JA": recall_score(y, p, pos_label=0, zero_division=0),
        "F1_JA": f1_score(y, p, pos_label=0, zero_division=0)})


def show(df, title=None, fmt=None, index=True):
    """Display a frame as a Markdown table; fmt maps column -> format function."""
    fmt = fmt or {}
    align = (["left"] if index else []) + ["right" if pd.api.types.is_numeric_dtype(df[c]) else "left"
                                           for c in df.columns]
    out = df.apply(lambda col: col.map(lambda v: "" if pd.isna(v) else fmt[col.name](v))
                   if col.name in fmt else col)
    display(Markdown((f"#### {title}\n\n" if title else "")
                     + out.to_markdown(index=index, colalign=align, disable_numparse=True)))


clf = joblib.load(MODEL_PATH)
ev = load_dir(EVAL_DIR)
ev["p_HE"] = clf.predict_proba(features(ev))[:, 1]
ev["pred"] = np.where(ev.p_HE > 0.5, "HE", "JA")
ev["correct"] = ev.lang == ev.pred

# Authors: an evaluation author is "seen" if any of their works is in training.
ev_works = pd.read_csv(os.path.join(EVAL_DIR, "works.csv"))
tr_works = pd.read_csv(os.path.join(TRAIN_DIR, "works.csv"))
ev["author"] = ev.work.map(ev_works.set_index("codeName").author)
ev["authors"] = np.where(ev.author.isin(tr_works.author), "seen", "unseen")



def word_counts(langs):
    """Words per language and in total, with their share, after the same cleaning."""
    n = langs.value_counts().reindex(["HE", "JA"])
    n.index = ["Hebrew", "Judeo-Arabic"]
    n["Total"] = n.sum()
    return pd.DataFrame({"words": n, "share": n / n["Total"]})


for name, langs in [("Evaluation", ev.lang), ("Training", load_dir(TRAIN_DIR).lang)]:
    show(word_counts(langs), f"{name} words", {"words": "{:,}".format, "share": "{:.1%}".format})

print(f"Total number of works:        {len(tr_works) + ev.work.nunique():>9}")
print(f"Total number of authors:      {len(set(tr_works.author) | set(ev.author)):>9}")

by_group = ev.groupby("authors").agg(authors=("author", "nunique"), works=("work", "nunique"))
unseen = sorted(ev.loc[ev.authors == "unseen", "author"].unique())
print(f"\nNumber of works used for training:                     {len(tr_works):>5}"
      f"  (by {tr_works.author.nunique()} authors)")
print(f"Number of works set aside for evaluation:              {ev.work.nunique():>5}")
print(f"Number of authors never seen by the model:             {len(unseen):>5}  ({', '.join(unseen)})")
print(f"Number of works by the authors never seen by the model:{by_group.works.get('unseen', 0):>5}")
show(by_group, "Evaluation works by author group")

#### Evaluation words

|              |     words |   share |
|:-------------|----------:|--------:|
| Hebrew       |   387,879 |   36.2% |
| Judeo-Arabic |   683,038 |   63.8% |
| Total        | 1,070,917 |  100.0% |

#### Training words

|              |     words |   share |
|:-------------|----------:|--------:|
| Hebrew       |   972,619 |   34.1% |
| Judeo-Arabic | 1,879,792 |   65.9% |
| Total        | 2,852,411 |  100.0% |

Total number of works:              110
Total number of authors:             36

Number of works used for training:                        70  (by 11 authors)


Number of works set aside for evaluation:                 40
Number of authors never seen by the model:                25  (Bahya ibn Paquda, David ben Joshua Maimonides, Dawud al-Muqammis, Hai Gaon, Hefetz ben Yatzliah, Hoter ben Solomon, Isaac ibn Barun, Isaac ibn Ghiyath, Joseph ben Simeon, Joseph ibn Aknin, Joshua ha-Nagid, Judah Halevi, Judah Hayyuj, Judah Rosh ha-Seder, Judah ibn Quraysh, Moses ibn Ezra, Mubashshir ha-Levi, Nathan Av ha-Yeshiva, Nethanel ben Isaiah, Nethanel ben al-Fayyumi, Nissim Gaon, Obadiah Maimonides, Samuel ben Ali, Samuel ben Hofni Gaon, Zerahiah ha-Levi)
Number of works by the authors never seen by the model:   31


#### Evaluation works by author group

| authors   |   authors |   works |
|:----------|----------:|--------:|
| seen      |         9 |       9 |
| unseen    |        25 |      31 |

## 1. Accuracy per work and per author

Worst first. *Pooled* accuracy counts every word equally, so large works weigh more; the *mean over works* counts every work equally.

In [2]:
per_work = (ev.groupby("work")
              .apply(lambda x: pd.Series({"author": x.author.iat[0], "authors": x.authors.iat[0],
                                          "words": len(x), "pct_HE": (x.lang == "HE").mean(),
                                          "accuracy": x.correct.mean()}), include_groups=False)
              .sort_values("accuracy"))
show(per_work, "Per work", {"pct_HE": "{:.1%}".format, "accuracy": "{:.1%}".format, "words": "{:,}".format})

per_author = (ev.groupby("author")
                .apply(lambda x: pd.Series({"works": x.work.nunique(), "words": len(x),
                                            "accuracy": x.correct.mean()}), include_groups=False)
                .sort_values("accuracy")
                .astype({"works": int, "words": int}))
show(per_author, "Per author", {"accuracy": "{:.1%}".format, "words": "{:,}".format})

print(f"pooled accuracy         {ev.correct.mean():.1%}")
print(f"mean accuracy over works {per_work.accuracy.mean():.1%}")

#### Per work

| work                     | author                       | authors   |   words |   pct_HE |   accuracy |
|:-------------------------|:-----------------------------|:----------|--------:|---------:|-----------:|
| IbnQuraysh_Aygrt         | Judah ibn Quraysh            | unseen    |  18,034 |    45.5% |      89.4% |
| SaadiaG_rvt              | Saadia Gaon                  | seen      |   1,376 |     6.5% |      91.6% |
| IbnBarun_spr             | Isaac ibn Barun              | unseen    |  22,941 |    21.8% |      92.6% |
| Hayyuj_spr               | Judah Hayyuj                 | unseen    |  17,356 |    40.1% |      93.4% |
| alHarizi_ms`y            | Judah al-Harizi              | seen      |   3,339 |    64.7% |      93.4% |
| NathanYesh_prvSH         | Nathan Av ha-Yeshiva         | unseen    |  57,872 |    41.5% |      93.6% |
| TanhumYeru_dnyAl         | Tanhum ha-Yerushalmi         | seen      |  10,328 |    24.9% |      93.7% |
| BenHofniG_mbvA           | Samuel ben Hofni Gaon        | unseen    |  18,532 |    64.8% |      94.1% |
| DavidBenAbraMaim_brASHyt | David ben Abraham Maimonides | seen      |   9,016 |    32.9% |      94.8% |
| IbnGhiyath_khlt          | Isaac ibn Ghiyath            | unseen    |  33,013 |    18.2% |      95.2% |
| Unk_rvt                  | Unknown                      | seen      |     555 |    33.0% |      95.3% |
| JoshNag_tSHvbvt          | Joshua ha-Nagid              | unseen    |  20,958 |    43.1% |      95.4% |
| IbnBalam_yhvSH`          | Judah ibn Balam              | seen      |   3,718 |    34.2% |      95.7% |
| BenHofniG_spr3           | Samuel ben Hofni Gaon        | unseen    |  13,657 |    51.2% |      95.8% |
| IbnEzra_h`yvnym          | Moses ibn Ezra               | unseen    |  36,237 |    16.1% |      95.9% |
| JudahRosh_spr            | Judah Rosh ha-Seder          | unseen    |   3,657 |    32.8% |      96.0% |
| BenHofniG_spr1           | Samuel ben Hofni Gaon        | unseen    |   5,542 |    42.8% |      96.1% |
| IbnAknin_htglvt          | Joseph ibn Aknin             | unseen    |  82,297 |    20.3% |      96.2% |
| NissimG_HmSHh            | Nissim Gaon                  | unseen    |  69,228 |    50.9% |      96.3% |
| HaiG_mSHpTy              | Hai Gaon                     | unseen    |  22,892 |    39.8% |      96.4% |
| BenHofniG_pyrvSH         | Samuel ben Hofni Gaon        | unseen    |  76,508 |    33.5% |      96.4% |
| MubashshirLev_hSHgvt     | Mubashshir ha-Levi           | unseen    |  13,378 |    22.7% |      96.5% |
| JudahHaLev_spr           | Judah Halevi                 | unseen    |  47,351 |    11.4% |      96.5% |
| HefBenYatz_spr           | Hefetz ben Yatzliah          | unseen    |  18,468 |    57.3% |      96.8% |
| ObasiahMaim_mAmr         | Obadiah Maimonides           | unseen    |   8,321 |    25.5% |      97.0% |
| alFayyumi_gn             | Nethanel ben al-Fayyumi      | unseen    |  27,741 |    19.7% |      97.1% |
| AbMaimonides_tSHvbvt     | Abraham Maimonides           | seen      |  37,745 |    48.1% |      97.2% |
| BenHofniG_spr2           | Samuel ben Hofni Gaon        | unseen    |   9,689 |    41.2% |      97.3% |
| BenHofniG_spr4           | Samuel ben Hofni Gaon        | unseen    |  10,750 |    25.5% |      97.5% |
| ZerahiahLev_SHyr         | Zerahiah ha-Levi             | unseen    |   8,703 |    27.3% |      97.6% |
| SamBenAli_Agrvt          | Samuel ben Ali               | unseen    |  20,341 |    70.5% |      97.7% |
| NethBenIsa_mAvr          | Nethanel ben Isaiah          | unseen    | 146,168 |    66.8% |      97.8% |
| DavidBenJoshMaim_mvrh    | David ben Joshua Maimonides  | unseen    |  14,222 |    18.6% |      97.8% |
| AbrBenSol_yhvSH`         | Abraham ben Solomon          | seen      |  23,262 |    74.8% |      97.9% |
| IbnPaquda_tvrt           | Bahya ibn Paquda             | unseen    |  88,575 |    14.7% |      98.1% |
| alMuqammis_`SHrym        | Dawud al-Muqammis            | unseen    |  26,428 |     0.4% |      98.2% |
| HotBenSol_SHAlvt         | Hoter ben Solomon            | unseen    |  16,152 |    20.1% |      98.3% |
| JosBenSimeon_Aygrt       | Joseph ben Simeon            | unseen    |  10,064 |    13.4% |      98.3% |
| BenHofniG_`SHr           | Samuel ben Hofni Gaon        | unseen    |   9,794 |     7.1% |      98.4% |
| Maim_bAvr                | Maimonides                   | seen      |   6,709 |     0.1% |      99.1% |

#### Per author

| author                       |   works |   words |   accuracy |
|:-----------------------------|--------:|--------:|-----------:|
| Judah ibn Quraysh            |       1 |  18,034 |      89.4% |
| Saadia Gaon                  |       1 |   1,376 |      91.6% |
| Isaac ibn Barun              |       1 |  22,941 |      92.6% |
| Judah Hayyuj                 |       1 |  17,356 |      93.4% |
| Judah al-Harizi              |       1 |   3,339 |      93.4% |
| Nathan Av ha-Yeshiva         |       1 |  57,872 |      93.6% |
| Tanhum ha-Yerushalmi         |       1 |  10,328 |      93.7% |
| David ben Abraham Maimonides |       1 |   9,016 |      94.8% |
| Isaac ibn Ghiyath            |       1 |  33,013 |      95.2% |
| Unknown                      |       1 |     555 |      95.3% |
| Joshua ha-Nagid              |       1 |  20,958 |      95.4% |
| Judah ibn Balam              |       1 |   3,718 |      95.7% |
| Moses ibn Ezra               |       1 |  36,237 |      95.9% |
| Judah Rosh ha-Seder          |       1 |   3,657 |      96.0% |
| Joseph ibn Aknin             |       1 |  82,297 |      96.2% |
| Nissim Gaon                  |       1 |  69,228 |      96.3% |
| Samuel ben Hofni Gaon        |       7 | 144,472 |      96.3% |
| Hai Gaon                     |       1 |  22,892 |      96.4% |
| Mubashshir ha-Levi           |       1 |  13,378 |      96.5% |
| Judah Halevi                 |       1 |  47,351 |      96.5% |
| Hefetz ben Yatzliah          |       1 |  18,468 |      96.8% |
| Obadiah Maimonides           |       1 |   8,321 |      97.0% |
| Nethanel ben al-Fayyumi      |       1 |  27,741 |      97.1% |
| Abraham Maimonides           |       1 |  37,745 |      97.2% |
| Zerahiah ha-Levi             |       1 |   8,703 |      97.6% |
| Samuel ben Ali               |       1 |  20,341 |      97.7% |
| Nethanel ben Isaiah          |       1 | 146,168 |      97.8% |
| David ben Joshua Maimonides  |       1 |  14,222 |      97.8% |
| Abraham ben Solomon          |       1 |  23,262 |      97.9% |
| Bahya ibn Paquda             |       1 |  88,575 |      98.1% |
| Dawud al-Muqammis            |       1 |  26,428 |      98.2% |
| Hoter ben Solomon            |       1 |  16,152 |      98.3% |
| Joseph ben Simeon            |       1 |  10,064 |      98.3% |
| Maimonides                   |       1 |   6,709 |      99.1% |

pooled accuracy         96.4%
mean accuracy over works 96.1%


## 2. Precision and recall for Hebrew and Judeo-Arabic

*Precision*: of the words labelled HE (or JA), how many really are. *Recall*: of the words that really are HE (or JA), how many were found. In works with under 1% Hebrew, a few errors weigh heavily on the Hebrew scores; those works are marked with `*`.

In [3]:
y, p = (ev.lang == "HE").astype(int), (ev.pred == "HE").astype(int)
report = pd.DataFrame(classification_report(y, p, target_names=["JA", "HE"], output_dict=True)).T
report.loc["accuracy"] = [np.nan, np.nan, report.loc["accuracy", "f1-score"], len(y)]
show(report, "All evaluation words", {"precision": "{:.3f}".format, "recall": "{:.3f}".format,
                                      "f1-score": "{:.3f}".format, "support": "{:,.0f}".format})

pr = ev.groupby("work").apply(scores, include_groups=False).sort_values("F1_HE")
pr.index = [w + (" *" if s < 0.01 else "") for w, s in zip(pr.index, pr.pct_HE)]
cols = ["words", "pct_HE", "P_HE", "R_HE", "F1_HE", "P_JA", "R_JA", "F1_JA"]
show(pr[cols], "Per work", {"words": "{:,.0f}".format, "pct_HE": "{:.1%}".format,
                             **{c: "{:.3f}".format for c in cols[2:]}})

#### All evaluation words

|              |   precision |   recall |   f1-score |   support |
|:-------------|------------:|---------:|-----------:|----------:|
| JA           |       0.980 |    0.964 |      0.972 |   683,038 |
| HE           |       0.938 |    0.965 |      0.951 |   387,879 |
| accuracy     |             |          |      0.964 | 1,070,917 |
| macro avg    |       0.959 |    0.964 |      0.962 | 1,070,917 |
| weighted avg |       0.965 |    0.964 |      0.964 | 1,070,917 |

#### Per work

|                          |   words |   pct_HE |   P_HE |   R_HE |   F1_HE |   P_JA |   R_JA |   F1_JA |
|:-------------------------|--------:|---------:|-------:|-------:|--------:|-------:|-------:|--------:|
| Maim_bAvr *              |   6,709 |     0.1% |  0.034 |  0.400 |   0.063 |  1.000 |  0.992 |   0.996 |
| alMuqammis_`SHrym *      |  26,428 |     0.4% |  0.139 |  0.747 |   0.235 |  0.999 |  0.983 |   0.991 |
| SaadiaG_rvt              |   1,376 |     6.5% |  0.410 |  0.663 |   0.506 |  0.976 |  0.934 |   0.954 |
| IbnBarun_spr             |  22,941 |    21.8% |  0.800 |  0.879 |   0.837 |  0.965 |  0.939 |   0.952 |
| JudahHaLev_spr           |  47,351 |    11.4% |  0.817 |  0.893 |   0.853 |  0.986 |  0.974 |   0.980 |
| TanhumYeru_dnyAl         |  10,328 |    24.9% |  0.860 |  0.894 |   0.877 |  0.964 |  0.952 |   0.958 |
| IbnGhiyath_khlt          |  33,013 |    18.2% |  0.809 |  0.961 |   0.879 |  0.991 |  0.950 |   0.970 |
| IbnEzra_h`yvnym          |  36,237 |    16.1% |  0.814 |  0.964 |   0.882 |  0.993 |  0.958 |   0.975 |
| IbnQuraysh_Aygrt         |  18,034 |    45.5% |  0.858 |  0.920 |   0.888 |  0.929 |  0.873 |   0.900 |
| BenHofniG_`SHr           |   9,794 |     7.1% |  0.842 |  0.960 |   0.897 |  0.997 |  0.986 |   0.991 |
| IbnAknin_htglvt          |  82,297 |    20.3% |  0.869 |  0.960 |   0.912 |  0.989 |  0.963 |   0.976 |
| Hayyuj_spr               |  17,356 |    40.1% |  0.936 |  0.897 |   0.916 |  0.933 |  0.959 |   0.946 |
| MubashshirLev_hSHgvt     |  13,378 |    22.7% |  0.902 |  0.948 |   0.924 |  0.984 |  0.970 |   0.977 |
| DavidBenAbraMaim_brASHyt |   9,016 |    32.9% |  0.885 |  0.967 |   0.924 |  0.983 |  0.939 |   0.960 |
| NathanYesh_prvSH         |  57,872 |    41.5% |  0.901 |  0.950 |   0.925 |  0.963 |  0.926 |   0.944 |
| alFayyumi_gn             |  27,741 |    19.7% |  0.890 |  0.976 |   0.931 |  0.994 |  0.970 |   0.982 |
| Unk_rvt                  |     555 |    33.0% |  0.898 |  0.967 |   0.932 |  0.983 |  0.946 |   0.964 |
| IbnBalam_yhvSH`          |   3,718 |    34.2% |  0.934 |  0.941 |   0.937 |  0.969 |  0.965 |   0.967 |
| IbnPaquda_tvrt           |  88,575 |    14.7% |  0.901 |  0.979 |   0.938 |  0.996 |  0.981 |   0.989 |
| JosBenSimeon_Aygrt       |  10,064 |    13.4% |  0.910 |  0.970 |   0.939 |  0.995 |  0.985 |   0.990 |
| JudahRosh_spr            |   3,657 |    32.8% |  0.934 |  0.946 |   0.940 |  0.973 |  0.967 |   0.970 |
| DavidBenJoshMaim_mvrh    |  14,222 |    18.6% |  0.920 |  0.965 |   0.942 |  0.992 |  0.981 |   0.986 |
| ObasiahMaim_mAmr         |   8,321 |    25.5% |  0.913 |  0.974 |   0.943 |  0.991 |  0.968 |   0.979 |
| JoshNag_tSHvbvt          |  20,958 |    43.1% |  0.949 |  0.945 |   0.947 |  0.958 |  0.962 |   0.960 |
| BenHofniG_pyrvSH         |  76,508 |    33.5% |  0.934 |  0.961 |   0.948 |  0.980 |  0.966 |   0.973 |
| alHarizi_ms`y            |   3,339 |    64.7% |  0.935 |  0.965 |   0.950 |  0.932 |  0.877 |   0.904 |
| BenHofniG_spr4           |  10,750 |    25.5% |  0.931 |  0.973 |   0.951 |  0.990 |  0.975 |   0.983 |
| BenHofniG_spr1           |   5,542 |    42.8% |  0.959 |  0.949 |   0.954 |  0.962 |  0.970 |   0.966 |
| BenHofniG_mbvA           |  18,532 |    64.8% |  0.971 |  0.938 |   0.954 |  0.892 |  0.948 |   0.919 |
| HaiG_mSHpTy              |  22,892 |    39.8% |  0.961 |  0.949 |   0.955 |  0.967 |  0.974 |   0.970 |
| HotBenSol_SHAlvt         |  16,152 |    20.1% |  0.955 |  0.959 |   0.957 |  0.990 |  0.989 |   0.989 |
| ZerahiahLev_SHyr         |   8,703 |    27.3% |  0.935 |  0.980 |   0.957 |  0.992 |  0.974 |   0.983 |
| BenHofniG_spr3           |  13,657 |    51.2% |  0.969 |  0.947 |   0.958 |  0.946 |  0.969 |   0.957 |
| NissimG_HmSHh            |  69,228 |    50.9% |  0.957 |  0.971 |   0.964 |  0.969 |  0.955 |   0.962 |
| BenHofniG_spr2           |   9,689 |    41.2% |  0.959 |  0.976 |   0.968 |  0.983 |  0.971 |   0.977 |
| AbMaimonides_tSHvbvt     |  37,745 |    48.1% |  0.971 |  0.971 |   0.971 |  0.973 |  0.973 |   0.973 |
| HefBenYatz_spr           |  18,468 |    57.3% |  0.963 |  0.982 |   0.973 |  0.975 |  0.950 |   0.962 |
| NethBenIsa_mAvr          | 146,168 |    66.8% |  0.982 |  0.985 |   0.983 |  0.970 |  0.963 |   0.967 |
| SamBenAli_Agrvt          |  20,341 |    70.5% |  0.983 |  0.984 |   0.983 |  0.962 |  0.959 |   0.960 |
| AbrBenSol_yhvSH`         |  23,262 |    74.8% |  0.987 |  0.985 |   0.986 |  0.955 |  0.963 |   0.959 |

## 3. Held-out accuracy: authors seen and unseen in training

Every evaluation work is held out. The question here is whether the model also generalises to **authors** it has never seen. Authors with a single work in the corpus appear only in evaluation, and Ben Hofni was kept out of training on purpose.

In [4]:
groups = {"authors unseen in training": ev.authors == "unseen",
          "↳ of which Ben Hofni": ev.author == "Samuel ben Hofni Gaon",
          "authors also in training": ev.authors == "seen",
          "all": ev.authors.notna()}
held = pd.DataFrame({name: scores(ev[m]) for name, m in groups.items()}).T
held.insert(1, "authors", [ev[m].author.nunique() for m in groups.values()])
held.insert(4, "share_of_words", held.words / len(ev))
held["AUC"] = [roc_auc_score(ev[m].lang == "HE", ev[m].p_HE) for m in groups.values()]
cols = ["works", "authors", "words", "share_of_words", "accuracy", "AUC", "F1_JA", "F1_HE"]
show(held[cols], fmt={"works": "{:.0f}".format, "authors": "{:.0f}".format,
                      "words": "{:,.0f}".format, "share_of_words": "{:.1%}".format,
                      "accuracy": "{:.1%}".format, "AUC": "{:.3f}".format,
                      "F1_JA": "{:.3f}".format, "F1_HE": "{:.3f}".format})

|                            |   works |   authors |     words |   share_of_words |   accuracy |   AUC |   F1_JA |   F1_HE |
|:---------------------------|--------:|----------:|----------:|-----------------:|-----------:|------:|--------:|--------:|
| authors unseen in training |      31 |        25 |   974,869 |            91.0% |      96.4% | 0.993 |   0.972 |   0.950 |
| ↳ of which Ben Hofni       |       7 |         1 |   144,472 |            13.5% |      96.3% | 0.993 |   0.971 |   0.952 |
| authors also in training   |       9 |         9 |    96,048 |             9.0% |      96.6% | 0.994 |   0.968 |   0.964 |
| all                        |      40 |        34 | 1,070,917 |           100.0% |      96.4% | 0.994 |   0.972 |   0.951 |

## 4. Confusion matrix

Rows are the true language, columns the model's label. Percentages are per row: the share of each language that was labelled correctly or not.

In [5]:
cm = pd.DataFrame(confusion_matrix(ev.lang, ev.pred, labels=["HE", "JA"]),
                  index=["true HE", "true JA"], columns=["labelled HE", "labelled JA"])
pct = cm.div(cm.sum(axis=1), axis=0)
show(cm, "Words", {c: "{:,}".format for c in cm.columns})
show(pct, "Share of each true language", {c: "{:.1%}".format for c in pct.columns})
print(f"Hebrew labelled Hebrew:   {cm.iat[0, 0]:>9,}")
print(f"Judeo-Arabic labelled JA: {cm.iat[1, 1]:>9,}")
print(f"Judeo-Arabic labelled HE: {cm.iat[1, 0]:>9,}")
print(f"Hebrew labelled JA:       {cm.iat[0, 1]:>9,}")

#### Words

|         |   labelled HE |   labelled JA |
|:--------|--------------:|--------------:|
| true HE |       374,330 |        13,549 |
| true JA |        24,760 |       658,278 |

#### Share of each true language

|         |   labelled HE |   labelled JA |
|:--------|--------------:|--------------:|
| true HE |         96.5% |          3.5% |
| true JA |          3.6% |         96.4% |

Hebrew labelled Hebrew:     374,330
Judeo-Arabic labelled JA:   658,278
Judeo-Arabic labelled HE:    24,760
Hebrew labelled JA:          13,549


## 5. Errors at language switches

A word is *at a switch* if the word before or after it on the same page is in the other language. A *run* is a stretch of consecutive words in the same language; short runs are, for example, a Hebrew term inside Arabic prose.

In [6]:
prev = ev.groupby("doc").lang.shift(1)
nxt = ev.groupby("doc").lang.shift(-1)
ev["at_switch"] = (prev.notna() & (prev != ev.lang)) | (nxt.notna() & (nxt != ev.lang))

sw = ev.groupby("at_switch").agg(words=("correct", "size"), error_rate=("correct", lambda c: 1 - c.mean()))
sw.index = sw.index.map({True: "at a switch", False: "inside a run"})
sw.insert(1, "share_of_words", sw.words / len(ev))
sw["share_of_errors"] = ev.groupby("at_switch").correct.apply(lambda c: (~c).sum()).values / (~ev.correct).sum()
show(sw, fmt={"words": "{:,}".format, "share_of_words": "{:.1%}".format,
              "error_rate": "{:.1%}".format, "share_of_errors": "{:.1%}".format})

# length of the run each word belongs to
run_id = ((ev.lang != prev) | (ev.doc != ev.doc.shift(1))).cumsum()
ev["run_len"] = ev.groupby(run_id).lang.transform("size")
ev["run"] = pd.cut(ev.run_len, [0, 1, 2, 5, 20, np.inf], labels=["1 word", "2", "3-5", "6-20", "over 20"])
runs = ev.groupby(["run", "lang"], observed=True).agg(words=("correct", "size"),
                                                      error_rate=("correct", lambda c: 1 - c.mean()))
runs = runs.unstack("lang")
runs.columns = [f"{stat} {lang}" for stat, lang in runs.columns]
show(runs, "Error rate by length of run", {"words HE": "{:,}".format, "words JA": "{:,}".format,
                                           "error_rate HE": "{:.1%}".format, "error_rate JA": "{:.1%}".format})

| at_switch    |   words |   share_of_words |   error_rate |   share_of_errors |
|:-------------|--------:|-----------------:|-------------:|------------------:|
| inside a run | 894,395 |            83.5% |         2.0% |             47.5% |
| at a switch  | 176,522 |            16.5% |        11.4% |             52.5% |

#### Error rate by length of run

| run     |   words HE |   words JA |   error_rate HE |   error_rate JA |
|:--------|-----------:|-----------:|----------------:|----------------:|
| 1 word  |     12,477 |      8,965 |           27.4% |           27.3% |
| 2       |     20,840 |     14,322 |            8.3% |           11.6% |
| 3-5     |     53,268 |     45,020 |            5.5% |            6.8% |
| 6-20    |    109,816 |    178,241 |            2.6% |            4.0% |
| over 20 |    191,478 |    436,490 |            1.4% |            2.4% |

## 6. Confidence

The model gives every word a probability of being Hebrew. Words near 0.5 are the ones it is unsure about. This shows how often it is wrong at each level of confidence.

In [7]:
bands = [0, 0.05, 0.2, 0.4, 0.6, 0.8, 0.95, 1.0001]
names = ["0-0.05 (sure JA)", "0.05-0.2", "0.2-0.4", "0.4-0.6 (unsure)", "0.6-0.8", "0.8-0.95", "0.95-1 (sure HE)"]
ev["band"] = pd.cut(ev.p_HE, bands, labels=names, right=False)
conf = ev.groupby("band", observed=True).agg(words=("correct", "size"),
                                             error_rate=("correct", lambda c: 1 - c.mean()))
conf.insert(1, "share_of_words", conf.words / len(ev))
show(conf, fmt={"words": "{:,}".format, "share_of_words": "{:.1%}".format, "error_rate": "{:.1%}".format})

unsure = ev.p_HE.between(0.2, 0.8)
print(f"unsure words (0.2-0.8): {unsure.mean():.1%} of all words, wrong {1 - ev.correct[unsure].mean():.1%} of the time")
print(f"all other words:        {(~unsure).mean():.1%} of all words, wrong {1 - ev.correct[~unsure].mean():.1%} of the time")
print(f"share of all errors among unsure words: {(~ev.correct[unsure]).sum() / (~ev.correct).sum():.1%}")

| band             |   words |   share_of_words |   error_rate |
|:-----------------|--------:|-----------------:|-------------:|
| 0-0.05 (sure JA) | 545,397 |            50.9% |         0.2% |
| 0.05-0.2         |  81,209 |             7.6% |         4.0% |
| 0.2-0.4          |  33,626 |             3.1% |        16.0% |
| 0.4-0.6 (unsure) |  22,357 |             2.1% |        43.2% |
| 0.6-0.8          |  27,291 |             2.5% |        36.1% |
| 0.8-0.95         |  59,380 |             5.5% |        10.4% |
| 0.95-1 (sure HE) | 301,657 |            28.2% |         0.9% |

unsure words (0.2-0.8): 7.8% of all words, wrong 29.9% of the time
all other words:        92.2% of all words, wrong 1.4% of the time
share of all errors among unsure words: 65.0%


## 7. Most frequent words: mislabelled, Judeo-Arabic, Hebrew

Words after cleaning (diacritics and punctuation removed). The first table shows how often each word occurs and how often the model gets it wrong, with one error of each kind: a Judeo-Arabic use labelled Hebrew (`example JA`) and a Hebrew use labelled Judeo-Arabic (`example HE`); the other two show the words the model weights most strongly towards each language. Each word is scored on its own, without its neighbours: `weight` is the sum of its chunk weights (negative = Judeo-Arabic, positive = Hebrew, as in section 8). Only words occurring at least 50 times are included. Each word comes with a five-word example (the word in bold) from the work where it is most frequent.

In [8]:
TOP = 5  # rows per table in sections 7 and 8

by_word = ev.groupby("w").agg(occurrences=("correct", "size"), errors=("correct", lambda c: (~c).sum()))
by_word["error_rate"] = by_word.errors / by_word.occurrences
fmt = {"occurrences": "{:,}".format, "errors": "{:,}".format, "error_rate": "{:.1%}".format}


def example(rows):
    """Five-word excerpt around one of the words in rows (a boolean mask over ev), in bold, and its source.
    The word is taken from the work with the most matching rows."""
    idx = ev.index[rows]
    if len(idx) == 0:
        return "", ""
    work = ev.work[idx].value_counts().index[0]
    for i in idx[ev.work[idx] == work]:
        win = ev.loc[i - 2:i + 2]
        if len(win) == 5 and win.doc.nunique() == 1:
            break
    text = " ".join(f"**{x}**" if j == i else x for j, x in zip(win.index, win.word))
    return text, f"{ev.author[i]}, {work}, p. {ev.doc[i].split(':')[1]}"


# examples of the errors: a Judeo-Arabic use labelled HE, and a Hebrew use labelled JA
mislabelled = by_word.sort_values("errors", ascending=False).head(TOP)
for lang in ["JA", "HE"]:
    mislabelled[[f"example {lang}", f"source {lang}"]] = [example((ev.w == w) & (ev.lang == lang) & ~ev.correct)
                                                          for w in mislabelled.index]
show(mislabelled, f"{TOP} most frequently mislabelled words", fmt)

# Each word scored alone (no neighbours): the sum of its chunk weights, as in section 8.
# Only words occurring at least 50 times, so that rare one-off spellings do not fill the lists.
counts = ev.w.value_counts()
counts = counts[counts >= 50]
word_weight = pd.DataFrame({"weight": clf.decision_function(" ~" + counts.index + "~ "),
                            "occurrences": counts.values}, index=counts.index.rename("w"))

for name, top in [("Judeo-Arabic", word_weight.nsmallest(TOP, "weight")),
                  ("Hebrew", word_weight.nlargest(TOP, "weight"))]:
    top[["example", "source"]] = [example(ev.w == w) for w in top.index]
    show(top, f"{TOP} words weighted most strongly towards {name}",
         {"weight": "{:.2f}".format, "occurrences": "{:,}".format})

#### 5 most frequently mislabelled words

| w   |   occurrences |   errors |   error_rate | example JA                        | source JA                                     | example HE                    | source HE                                   |
|:----|--------------:|---------:|-------------:|:----------------------------------|:----------------------------------------------|:------------------------------|:--------------------------------------------|
| לא  |        10,363 |      735 |         7.1% | כבית שמאי **לא** כקול בית         | Nathan Av ha-Yeshiva, NathanYesh_prvSH, p. 13 | כמי קאל **לא** תוהו בראה      | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ז  |
| מן  |        20,553 |      504 |         2.5% | והו משתק **מן** שמרים ממחים       | Judah Hayyuj, Hayyuj_spr, p. 93               | הגן ולא **מן** הגזל. על       | Nethanel ben Isaiah, NethBenIsa_mAvr, p. מג |
| ולא |         5,831 |      433 |         7.4% | מות'וק בצדקה, **ולא** תאנס אלנפוס | Bahya ibn Paquda, IbnPaquda_tvrt, p. לא       | ביום ראשון **ולא** תלאן עד    | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ז  |
| או  |         3,405 |      309 |         9.1% | דם נדה **או** כל רגל              | Nathan Av ha-Yeshiva, NathanYesh_prvSH, p. 7  | היה חגר **או** חולה. וישלחה,  | Nethanel ben Isaiah, NethBenIsa_mAvr, p. קג |
| אלא |         3,918 |      295 |         7.5% | לחתי אשכר **אלא** למת'ל קולה      | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ד    | אינו מכנען **אלא** תגר. ויקרא | Nethanel ben Isaiah, NethBenIsa_mAvr, p. קס |

#### 5 words weighted most strongly towards Judeo-Arabic

| w      |   weight |   occurrences | example                             | source                                     |
|:-------|---------:|--------------:|:------------------------------------|:-------------------------------------------|
| אן     |   -19.71 |        13,860 | תפסיר ד'אלך **אן** כל אלמכ'לוקאת    | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ה |
| אלדי   |   -16.74 |         4,870 | מנהא אלרתבה **אלד'י** פיהא אנטבאע   | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א |
| אלמתל  |   -16.49 |            63 | ואנמא צ'רבת **אלמת'ל** ביין מן      | Joseph ibn Aknin, IbnAknin_htglvt, p. 26   |
| אלמא   |   -16.10 |           347 | אלטביעי, פכאן **אלמא** מחיט באלארץ' | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ג |
| אלמעני |   -15.85 |           701 | קול יעצ'ד **אלמעני** אלד'י ד'כרתה   | Isaac ibn Barun, IbnBarun_spr, p. 4        |

#### 5 words weighted most strongly towards Hebrew

| w     |   weight |   occurrences | example                        | source                                       |
|:------|---------:|--------------:|:-------------------------------|:---------------------------------------------|
| ה     |    15.95 |         1,593 | עזרי מעם **ה'** עושה שמים      | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א   |
| זל    |    15.29 |         1,127 | פיה רבותינו **ז"ל** וחאשאה אן  | Joseph ibn Aknin, IbnAknin_htglvt, p. 2      |
| ר     |    14.04 |         1,861 | [רואה] ען **ר'** יוחנן ואלרואה | Samuel ben Hofni Gaon, BenHofniG_mbvA, p. 23 |
| ישראל |    13.22 |         1,787 | מה יעשה **ישראל** וליס הד'י    | Nethanel ben Isaiah, NethBenIsa_mAvr, p. כב  |
| השמים |    12.84 |           230 | קאל את **השמים,** יעני אלמאדה  | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ו   |

## 8. Most frequent chunks: mislabelled, Judeo-Arabic, Hebrew

The model does not see whole words: it sees **chunks** of 1 to 5 letters taken from the word and its neighbours, each with a learned weight. Here the chunks are taken from each word on its own, with `_` marking the start or end of the word (so `_אל` is a word beginning with אל). Single letters are left out, as they occur everywhere.

The last two tables show the chunks the model itself weights most strongly towards each language, which is the most direct view of what it has learned.

Each chunk comes with a five-word example from the work where it is most frequent, with a word containing the chunk in bold: in the first table, one error of each kind (a Judeo-Arabic use labelled Hebrew, and a Hebrew use labelled Judeo-Arabic); in the others, a word in the language of the table.

In [9]:
vec, lr = clf.steps[0][1], clf.steps[-1][1]
analyse = vec.build_analyzer()


def chunk_counts(words):
    """Count the chunks (2-5 letters) in a Series of words, weighted by occurrences."""
    counts = Counter()
    for w, n in words.value_counts().items():
        for c in set(analyse(w)):
            if len(c.strip()) >= 2:
                counts[c.replace(" ", "_")] += n
    return counts


unique_words = pd.Series(ev.w.unique())


def has_chunk(chunk, edge):
    """Mask over ev: words containing chunk, where edge (_ or ~) marks the start or end of the word."""
    pattern = "^" * chunk.startswith(edge) + re.escape(chunk.strip(edge)) + "$" * chunk.endswith(edge)
    return ev.w.isin(unique_words[unique_words.str.contains(pattern)])


all_chunks = chunk_counts(ev.w)
err_chunks = chunk_counts(ev.w[~ev.correct])
top_err = pd.DataFrame(err_chunks.most_common(TOP), columns=["chunk", "in mislabelled words"]).set_index("chunk")
top_err["in all words"] = [all_chunks[c] for c in top_err.index]
top_err["error_rate"] = top_err["in mislabelled words"] / top_err["in all words"]
for lang in ["JA", "HE"]:
    top_err[[f"example {lang}", f"source {lang}"]] = [example(has_chunk(k, "_") & (ev.lang == lang) & ~ev.correct)
                                                      for k in top_err.index]
show(top_err, f"{TOP} most frequent chunks in mislabelled words",
     {"in mislabelled words": "{:,}".format, "in all words": "{:,}".format, "error_rate": "{:.1%}".format})

he_chunks = chunk_counts(ev.w[ev.lang == "HE"])
for lang, name in [("JA", "Judeo-Arabic"), ("HE", "Hebrew")]:
    c = chunk_counts(ev.w[ev.lang == lang])
    top = pd.DataFrame(c.most_common(TOP), columns=["chunk", f"in {lang} words"]).set_index("chunk")
    top["share_HE"] = [he_chunks[k] / all_chunks[k] for k in top.index]
    top[["example", "source"]] = [example(has_chunk(k, "_") & (ev.lang == lang)) for k in top.index]
    show(top, f"{TOP} most frequent chunks in {name} words (share_HE: how often the chunk occurs in Hebrew words)",
         {f"in {lang} words": "{:,}".format, "share_HE": "{:.1%}".format})

# strongest chunks according to the model (target-word chunks, marked with ~)
names, coef = vec.get_feature_names_out(), lr.coef_[0]
order = np.argsort(coef)


def strongest(idx, n=TOP):
    """First n target-word chunks in idx, counting padded variants (' ~אל', '~אל') once."""
    out, seen = [], set()
    for i in idx:
        key = names[i].strip()
        if "~" in key and key not in seen:
            seen.add(key)
            out.append((key, round(coef[i], 2)))
        if len(out) == n:
            break
    return pd.DataFrame(out, columns=["chunk", "weight"])


for lang, label, idx in [("JA", "Judeo-Arabic", order), ("HE", "Hebrew", order[::-1])]:
    top = strongest(idx)
    top[["example", "source"]] = [example(has_chunk(k, "~") & (ev.lang == lang)) for k in top.chunk]
    show(top, f"{TOP} chunks weighted most strongly towards {label} (~ marks the edge of the word being labelled)",
         {"weight": "{:.2f}".format}, index=False)

#### 5 most frequent chunks in mislabelled words

| chunk   |   in mislabelled words |   in all words |   error_rate | example JA                               | source JA                                     | example HE                           | source HE                                      |
|:--------|-----------------------:|---------------:|-------------:|:-----------------------------------------|:----------------------------------------------|:-------------------------------------|:-----------------------------------------------|
| לא      |                  2,150 |         59,845 |         3.6% | לחתי אשכר **אלא** למת'ל קולה             | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ד    | כמי קאל **לא** תוהו בראה             | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ז     |
| אל      |                  2,075 |        180,307 |         1.2% | האדם וחי **אלא** תרי קול                 | Joseph ibn Aknin, IbnAknin_htglvt, p. 10      | בעד אלולאד. **ואליך** תשוקתו, אלשכ'ץ | Nethanel ben Isaiah, NethBenIsa_mAvr, p. נא    |
| לא_     |                  1,687 |         25,858 |         6.5% | לחתי אשכר **אלא** למת'ל קולה             | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ד    | כמי קאל **לא** תוהו בראה             | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ז     |
| לי      |                  1,659 |         46,373 |         3.6% | אלמסמי דמנה **וכלילה** אלגזתה במכ'אטאבאת | Joseph ibn Aknin, IbnAknin_htglvt, p. 2       | פקאל לבן **ליעק'** מא הד'א           | Samuel ben Hofni Gaon, BenHofniG_pyrvSH, p. יא |
| ול      |                  1,500 |         38,211 |         3.9% | אעני תודה **בקול'** וכי תזבחו            | Nathan Av ha-Yeshiva, NathanYesh_prvSH, p. 17 | ביום ראשון **ולא** תלאן עד           | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ז     |

#### 5 most frequent chunks in Judeo-Arabic words (share_HE: how often the chunk occurs in Hebrew words)

| chunk   |   in JA words |   share_HE | example                                  | source                                  |
|:--------|--------------:|-----------:|:-----------------------------------------|:----------------------------------------|
| אל      |       168,286 |       6.7% | מקדמה כתאב **אלהדאיה** אלי פראיץ'        | Bahya ibn Paquda, IbnPaquda_tvrt, p. יג |
| _אל     |       125,088 |       5.1% | מקדמה כתאב **אלהדאיה** אלי פראיץ'        | Bahya ibn Paquda, IbnPaquda_tvrt, p. יג |
| לא      |        47,097 |      21.3% | אלואחד, אלחק, **אלאזלי** אלוג'וד, אלדאים | Bahya ibn Paquda, IbnPaquda_tvrt, p. יג |
| וא      |        41,597 |      21.4% | אלחקיק במעני **אלואחד,** אלחק, אלאזלי    | Bahya ibn Paquda, IbnPaquda_tvrt, p. יג |
| לי      |        34,975 |      24.6% | כתאב אלהדאיה **אלי** פראיץ' אלקלוב       | Bahya ibn Paquda, IbnPaquda_tvrt, p. יג |

#### 5 most frequent chunks in Hebrew words (share_HE: how often the chunk occurs in Hebrew words)

| chunk   |   in HE words |   share_HE | example                            | source                                     |
|:--------|--------------:|-----------:|:-----------------------------------|:-------------------------------------------|
| ים      |        20,349 |      83.8% | ה' עושה **שמים** וארץ. אשכילך      | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א |
| ים_     |        20,344 |      83.8% | ה' עושה **שמים** וארץ. אשכילך      | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א |
| ות      |        17,035 |      70.7% | וענני ומכל **מגורותי** הצילני. קאל | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א |
| ני      |        14,707 |      54.3% | איעצה עליך **עיני.** דרשתי את      | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א |
| לא      |        12,748 |      21.3% | התורה בבי **ולא** התחיל באלף       | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ה |

#### 5 chunks weighted most strongly towards Judeo-Arabic (~ marks the edge of the word being labelled)

| chunk   |   weight | example                               | source                                        |
|:--------|---------:|:--------------------------------------|:----------------------------------------------|
| ~אל     |   -18.58 | מקדמה כתאב **אלהדאיה** אלי פראיץ'     | Bahya ibn Paquda, IbnPaquda_tvrt, p. יג       |
| ~הו~    |   -13.72 | וד'לך אלפיץ' **הו** כמאלהא אלאכ'יר    | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ב    |
| הא~     |   -11.45 | לכתאבה אלעזיז **פג'עלוהא** פי אלמנזלה | Joseph ibn Aknin, IbnAknin_htglvt, p. 2       |
| קולה~   |   -11.16 | ואלספר אלב' **קולה** לה ואתה          | Samuel ben Hofni Gaon, BenHofniG_pyrvSH, p. ז |
| דלך~    |   -10.75 | ויוציאם בכסף **וכד'לך** פימא יאתי     | Samuel ben Hofni Gaon, BenHofniG_pyrvSH, p. ג |

#### 5 chunks weighted most strongly towards Hebrew (~ marks the edge of the word being labelled)

| chunk   |   weight | example                           | source                                        |
|:--------|---------:|:----------------------------------|:----------------------------------------------|
| ~ה      |    11.97 | עזרי מעם **ה'** עושה שמים         | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א    |
| ~אלא~   |    11.14 | ואין אחד **אלא** הקב"ה אבל        | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ה    |
| אלא~    |     9.15 | ואין אחד **אלא** הקב"ה אבל        | Nethanel ben Isaiah, NethBenIsa_mAvr, p. ה    |
| יהו~    |     8.23 | מעשיהם: וספר **ירמיהו** פאנה סאפר | Samuel ben Hofni Gaon, BenHofniG_pyrvSH, p. ה |
| ים~     |     7.85 | ה' עושה **שמים** וארץ. אשכילך     | Nethanel ben Isaiah, NethBenIsa_mAvr, p. א    |

## 9. Words analysis: errors, Judeo-Arabic, Hebrew

The words of section 7 in one table: the words most often mislabelled, sorted by their error rate, and the words weighted most strongly towards each language with their weight. Markdown cannot merge header cells, so this one table is HTML.

In [10]:
errors = mislabelled.sort_values("error_rate", ascending=False)
ja, he = word_weight.nsmallest(TOP, "weight"), word_weight.nlargest(TOP, "weight")
analysis = pd.DataFrame({("Errors", "word"): errors.index, ("Errors", "error rate"): errors.error_rate.values,
                         ("Judeo-Arabic words", "word"): ja.index, ("Judeo-Arabic words", "weight"): ja.weight.values,
                         ("Hebrew words", "word"): he.index, ("Hebrew words", "weight"): he.weight.values})
analysis = pd.concat({"Words analysis": analysis}, axis=1)  # top header row spanning the whole table

display(analysis.style.hide(axis="index")
        .format({("Words analysis", "Errors", "error rate"): "{:.1%}",
                 ("Words analysis", "Judeo-Arabic words", "weight"): "{:.2f}",
                 ("Words analysis", "Hebrew words", "weight"): "{:.2f}"})
        .set_table_styles([{"selector": "th.col_heading.level0, th.col_heading.level1",
                            "props": "text-align: center"}]))